# 🃏 TFG — Royal FLush: Introducción a SPADE y Agentes

---

## 🎯 Objetivo de este notebook

Este notebook es el punto de partida de tu TFG. Aquí vas a:

1. Entender por qué Python `asyncio` es la base de todo
2. Crear tu primer agente SPADE desde cero
3. Comunicar dos agentes entre sí
4. Implementar una Máquina de Estados Finita (FSM)
5. Relacionar todo con el código real de Royal FLush

> ⚠️ **Requisito:** Para ejecutar los agentes SPADE necesitas un servidor XMPP local.  
> La forma más fácil es instalar **PyJabber** (servidor XMPP en Python):  
> ```bash
> pip install pyjabber
> pyjabber  # lo lanza en localhost:5222
> ```  
> Las secciones de `asyncio` puro (1 y 2) no lo necesitan.

In [5]:
#!pip install spade

  Attempting uninstall: SQLAlchemy
    Found existing installation: SQLAlchemy 1.4.22



ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
conda-repo-cli 1.0.4 requires pathlib, which is not installed.
cookiecutter 1.7.2 requires Jinja2<3.0.0, but you have jinja2 3.0.3 which is incompatible.
cookiecutter 1.7.2 requires MarkupSafe<2.0.0, but you have markupsafe 2.1.5 which is incompatible.


---
## 📦 Instalación de dependencias

Ejecuta esta celda una sola vez para instalar todo lo necesario.

In [1]:
# Instala las dependencias del proyecto
# Si ya tienes el entorno configurado, puedes saltar esta celda
#import sys

#!{sys.executable} -m pip install spade==4.1.2 pyjabber --quiet
#print("✅ Dependencias instaladas")

✅ Dependencias instaladas


"C:\Users\asus" no se reconoce como un comando interno o externo,
programa o archivo por lotes ejecutable.


---
# PARTE 1 — La base de todo: `asyncio` 🔄

## ¿Por qué asyncio?

SPADE usa `asyncio` para ejecutar múltiples agentes de forma **concurrente** en un solo hilo.  
Es el mismo modelo que usan los servidores web modernos.

La idea clave es que, en lugar de bloquear el programa esperando (ej: a que llegue un mensaje),  
usamos `await` para decir: *"Pause aquí y deja que otro agente trabaje mientras espero"*.

```
MODELO SÍNCRONO (bloqueante):      MODELO ASYNCIO (concurrente):

Agente A: ████░░░░████░░░░          Agente A: ████░░████░░████
Agente B: ░░░░████░░░░████          Agente B: ░░██░░░░██░░░░██
          ─────────────────                   ─────────────────
          tiempo de CPU desperdiciado          CPU siempre ocupada
```

### Conceptos clave:
| Concepto | Qué es |
|---|---|
| `async def` | Define una **corrutina** (función que puede pausarse) |
| `await` | Pausa la corrutina hasta que otra termina |
| `asyncio.sleep()` | Pausa SIN bloquear (deja actuar a otros) |
| `asyncio.gather()` | Ejecuta varias corrutinas a la vez |

In [1]:
import asyncio
import time

# ──────────────────────────────────────────────────────────────
# EJEMPLO 1: La diferencia entre time.sleep y asyncio.sleep
# ──────────────────────────────────────────────────────────────

async def agente_bloqueante(nombre: str, segundos: int):
    """Simula un agente que BLOQUEA el hilo (MAL ejemplo)."""
    print(f"[{nombre}] Empezando tarea de {segundos}s...")
    time.sleep(segundos)  # ❌ Esto bloquea TODO el programa
    print(f"[{nombre}] ¡Terminé! (bloqueante)")

async def agente_async(nombre: str, segundos: int):
    """Simula un agente que COOPERA con otros (BUEN ejemplo)."""
    print(f"[{nombre}] Empezando tarea de {segundos}s...")
    await asyncio.sleep(segundos)  # ✅ Pausa y deja actuar a otros
    print(f"[{nombre}] ¡Terminé! (async)")

print("=== VERSIÓN BLOQUEANTE (secuencial) ===")
inicio = time.time()
await agente_bloqueante("Agente-A", 2)
await agente_bloqueante("Agente-B", 2)
print(f"Tiempo total: {time.time() - inicio:.1f}s\n")

print("=== VERSIÓN ASYNC (concurrente) ===")
inicio = time.time()
await asyncio.gather(
    agente_async("Agente-A", 2),
    agente_async("Agente-B", 2),
)
print(f"Tiempo total: {time.time() - inicio:.1f}s")

=== VERSIÓN BLOQUEANTE (secuencial) ===
[Agente-A] Empezando tarea de 2s...
[Agente-A] ¡Terminé! (bloqueante)
[Agente-B] Empezando tarea de 2s...
[Agente-B] ¡Terminé! (bloqueante)
Tiempo total: 4.0s

=== VERSIÓN ASYNC (concurrente) ===
[Agente-A] Empezando tarea de 2s...
[Agente-B] Empezando tarea de 2s...
[Agente-A] ¡Terminé! (async)
[Agente-B] ¡Terminé! (async)
Tiempo total: 2.0s


**¿Qué observas?** La versión bloqueante tarda ~4s (2+2 en secuencia).  
La versión async tarda ~2s porque ambos agentes "esperan" al mismo tiempo.

Esto es exactamente lo que hace Royal FLush: varios agentes entrenando, comunicándose  
y esperando mensajes al mismo tiempo, sin bloquearse entre sí.

In [2]:
import asyncio

# ──────────────────────────────────────────────────────────────
# EJEMPLO 2: Queue asyncio — la herramienta de PACoL
# ──────────────────────────────────────────────────────────────
# En Royal FLush, el LayerReceiverState guarda modelos recibidos
# en una asyncio.Queue. El ConsensusState los consume.
# Aquí simulamos eso.

async def productor(cola: asyncio.Queue, nombre: str):
    """Simula un agente que ENVÍA pesos de su modelo."""
    for ronda in range(1, 4):
        modelo_simulado = {"fc1.weight": ronda * 0.1, "fc2.weight": ronda * 0.2}
        print(f"[{nombre}] → Enviando modelo de ronda {ronda}: {modelo_simulado}")
        await cola.put(modelo_simulado)  # Mete en la cola
        await asyncio.sleep(0.5)         # Simula tiempo de entrenamiento

async def consumidor(cola: asyncio.Queue, nombre: str):
    """Simula el ConsensusState que PROCESA los modelos recibidos."""
    modelos_recibidos = []
    while True:
        try:
            # Espera máximo 1s a que llegue algo
            modelo = await asyncio.wait_for(cola.get(), timeout=1.0)
            modelos_recibidos.append(modelo)
            print(f"[{nombre}] ← Recibido modelo: {modelo}")
            
            # Simula consenso: promedio de pesos
            if len(modelos_recibidos) >= 2:
                pesos_fc1 = [m["fc1.weight"] for m in modelos_recibidos]
                consenso = sum(pesos_fc1) / len(pesos_fc1)
                print(f"[{nombre}] 🤝 Consenso fc1.weight = {consenso:.3f}")
                modelos_recibidos.clear()
        except asyncio.TimeoutError:
            print(f"[{nombre}] No hay más modelos. Terminando.")
            break

# Ejecutar productor y consumidor EN PARALELO (como PACoL)
cola_compartida = asyncio.Queue()
await asyncio.gather(
    productor(cola_compartida, "Agente-A"),
    consumidor(cola_compartida, "Agente-B"),
)

[Agente-A] → Enviando modelo de ronda 1: {'fc1.weight': 0.1, 'fc2.weight': 0.2}
[Agente-B] ← Recibido modelo: {'fc1.weight': 0.1, 'fc2.weight': 0.2}
[Agente-A] → Enviando modelo de ronda 2: {'fc1.weight': 0.2, 'fc2.weight': 0.4}
[Agente-B] ← Recibido modelo: {'fc1.weight': 0.2, 'fc2.weight': 0.4}
[Agente-B] 🤝 Consenso fc1.weight = 0.150
[Agente-A] → Enviando modelo de ronda 3: {'fc1.weight': 0.30000000000000004, 'fc2.weight': 0.6000000000000001}
[Agente-B] ← Recibido modelo: {'fc1.weight': 0.30000000000000004, 'fc2.weight': 0.6000000000000001}
[Agente-B] No hay más modelos. Terminando.


[None, None]

**Conexión con Royal FLush:**  
En [`royalflush/behaviour/ppremiofl/layer_receiver.py`], la clase `ParallelLayerReceiverBehaviour`  
hace exactamente esto: recibe modelos de vecinos y los mete en `self.agent.consensus_transmissions` (una `Queue`).

El `ConsensusState` los consume cuando le toca. Así es el **paralelismo de PACoL**.

---
# PARTE 2 — Tu primer agente SPADE 🤖

## ¿Qué es un agente SPADE?

Un agente SPADE es un objeto Python que:
- Tiene un **JID** (identificador, como un email: `nombre@servidor`)
- Se comunica por **XMPP** (protocolo de mensajería)
- Ejecuta **Behaviours** (comportamientos) de forma asíncrona

### Tipos de Behaviours principales:

| Behaviour | Cuándo se ejecuta |
|---|---|
| `OneShotBehaviour` | Una sola vez y termina |
| `CyclicBehaviour` | En bucle infinito |
| `PeriodicBehaviour` | Cada N segundos |
| `FSMBehaviour` | Máquina de Estados Finita |

> ⚠️ **Necesitas PyJabber corriendo** para ejecutar las siguientes celdas.  
> Abre una terminal y ejecuta: `pyjabber`

In [4]:
import asyncio
import spade
from spade.agent import Agent
from spade.behaviour import OneShotBehaviour, CyclicBehaviour

# ──────────────────────────────────────────────────────────────
# AGENTE MÁS SIMPLE POSIBLE
# ──────────────────────────────────────────────────────────────

class MiPrimerBehaviour(OneShotBehaviour):
    """
    OneShotBehaviour: se ejecuta UNA VEZ y termina.
    Equivale a la fase de inicialización en Royal FLush.
    """
    async def run(self):
        # 'self.agent' da acceso al agente que contiene este behaviour
        print(f"[{self.agent.jid}] 👋 ¡Hola! Soy un agente SPADE.")
        print(f"[{self.agent.jid}] Mi JID completo es: {self.agent.jid}")
        print(f"[{self.agent.jid}] Voy a detenerme...")
        await self.agent.stop()  # Para el agente cuando termina


class MiPrimerAgente(Agent):
    """
    La clase Agent es la base. En Royal FLush se llama AgentBase.
    El método setup() es async y se llama al arrancar el agente.
    """
    async def setup(self):
        print(f"[{self.jid}] ⚙️  Configurando agente...")
        # Aquí se añaden los Behaviours al agente
        mi_behaviour = MiPrimerBehaviour()
        self.add_behaviour(mi_behaviour)
        print(f"[{self.jid}] ✅ Behaviour añadido.")


async def main_agente_simple():
    # Credenciales: JID y contraseña (el servidor XMPP crea la cuenta automáticamente)
    agente = MiPrimerAgente(
        jid="agente1@localhost",
        password="password123"
    )
    
    print("🚀 Iniciando agente...")
    await agente.start(auto_register=True)  # auto_register crea la cuenta en el servidor
    
    # Espera a que el agente termine
    while agente.is_alive():
        await asyncio.sleep(1)
    
    print("🏁 Agente finalizado.")
    # En esta versión no existe esta función (await agente.wait_until_finished())

# En Jupyter usamos await directamente (ya hay un event loop corriendo)
await main_agente_simple()

🚀 Iniciando agente...
[agente1@localhost] ⚙️  Configurando agente...
[agente1@localhost] ✅ Behaviour añadido.
[agente1@localhost] 👋 ¡Hola! Soy un agente SPADE.
[agente1@localhost] Mi JID completo es: agente1@localhost
[agente1@localhost] Voy a detenerme...
🏁 Agente finalizado.


**Conexión con Royal FLush:**  
En [`royalflush/agent/base.py`], la clase `AgentBase` hereda de `Agent` igual que arriba.  
Su `setup()` configura los manejadores de presencia XMPP, que son el mecanismo para  
saber si los vecinos están "online" (disponibles para recibir modelos).

---
## 2.2 — CyclicBehaviour: el corazón del entrenamiento

El `CyclicBehaviour` se ejecuta en bucle mientras el agente esté vivo.  
En Royal FLush, el bucle de entrenamiento (Train → Communicate → Consensus) es el ejemplo perfecto.

In [5]:
import asyncio
from spade.agent import Agent
from spade.behaviour import CyclicBehaviour

# ──────────────────────────────────────────────────────────────
# AGENTE CON BUCLE DE ENTRENAMIENTO SIMULADO
# ──────────────────────────────────────────────────────────────

class BucleEntrenamiento(CyclicBehaviour):
    """
    Simula el ciclo de vida de un Node en Royal FLush:
    Entrenar → Comunicar → Consenso → (repetir)
    """
    def __init__(self, max_rondas: int = 3):
        super().__init__()
        self.ronda_actual = 0
        self.max_rondas = max_rondas
        # Simula los pesos del modelo (en RF son tensores de PyTorch)
        self.pesos_modelo = {"capa1": 0.5, "capa2": 0.3}

    async def run(self):
        """Este método se llama en cada iteración del bucle."""
        
        if self.ronda_actual >= self.max_rondas:
            print(f"[{self.agent.jid.node}] 🏁 Alcanzado máximo de rondas ({self.max_rondas}). Parando.")
            await self.agent.stop()
            return

        self.ronda_actual += 1
        print(f"\n[{self.agent.jid.node}] ══ RONDA {self.ronda_actual}/{self.max_rondas} ══")
        
        # ── FASE 1: ENTRENAMIENTO LOCAL ──
        await self._fase_entrenamiento()
        
        # ── FASE 2: COMUNICACIÓN ──
        await self._fase_comunicacion()
        
        # ── FASE 3: CONSENSO ──
        await self._fase_consenso()

    async def _fase_entrenamiento(self):
        print(f"  [Train] 🧠 Entrenando modelo local...")
        await asyncio.sleep(0.2)  # En RF: forward + backward pass de PyTorch
        # Simula mejora del modelo tras cada época
        self.pesos_modelo = {k: v + 0.01 * self.ronda_actual for k, v in self.pesos_modelo.items()}
        print(f"  [Train] ✅ Modelo actualizado: {self.pesos_modelo}")

    async def _fase_comunicacion(self):
        vecino_simulado = "agente2@localhost"
        print(f"  [Comm]  📤 Enviando pesos a {vecino_simulado}...")
        await asyncio.sleep(0.1)  # En RF: send() via XMPP
        print(f"  [Comm]  ✅ Pesos enviados.")

    async def _fase_consenso(self):
        print(f"  [Cons]  🤝 Procesando modelos recibidos...")
        await asyncio.sleep(0.1)  # En RF: promedia los tensores de la Queue
        # Simula consenso: promedio de mi modelo con el recibido
        modelo_vecino = {"capa1": 0.4, "capa2": 0.6}  # Simulado
        self.pesos_modelo = {
            k: (self.pesos_modelo[k] + modelo_vecino[k]) / 2
            for k in self.pesos_modelo
        }
        print(f"  [Cons]  ✅ Modelo tras consenso: {self.pesos_modelo}")


class AgenteEntrenamiento(Agent):
    async def setup(self):
        bucle = BucleEntrenamiento(max_rondas=3)
        self.add_behaviour(bucle)


async def main_entrenamiento():
    agente = AgenteEntrenamiento("agente1@localhost", "password123")
    await agente.start(auto_register=True)
    while agente.is_alive():
        await asyncio.sleep(1)
    
    print("🏁 Agente finalizado.")#await agente.wait_until_finished()

await main_entrenamiento()


[agente1] ══ RONDA 1/3 ══
  [Train] 🧠 Entrenando modelo local...
  [Train] ✅ Modelo actualizado: {'capa1': 0.51, 'capa2': 0.31}
  [Comm]  📤 Enviando pesos a agente2@localhost...
  [Comm]  ✅ Pesos enviados.
  [Cons]  🤝 Procesando modelos recibidos...
  [Cons]  ✅ Modelo tras consenso: {'capa1': 0.455, 'capa2': 0.45499999999999996}

[agente1] ══ RONDA 2/3 ══
  [Train] 🧠 Entrenando modelo local...
  [Train] ✅ Modelo actualizado: {'capa1': 0.47500000000000003, 'capa2': 0.475}
  [Comm]  📤 Enviando pesos a agente2@localhost...
  [Comm]  ✅ Pesos enviados.
  [Cons]  🤝 Procesando modelos recibidos...
  [Cons]  ✅ Modelo tras consenso: {'capa1': 0.4375, 'capa2': 0.5375}

[agente1] ══ RONDA 3/3 ══
  [Train] 🧠 Entrenando modelo local...
  [Train] ✅ Modelo actualizado: {'capa1': 0.4675, 'capa2': 0.5675}
  [Comm]  📤 Enviando pesos a agente2@localhost...
  [Comm]  ✅ Pesos enviados.
  [Cons]  🤝 Procesando modelos recibidos...
  [Cons]  ✅ Modelo tras consenso: {'capa1': 0.43375, 'capa2': 0.58375}
[agent

---
# PARTE 3 — Comunicación entre agentes 📨

## Mensajes en SPADE

Los agentes se comunican mediante **mensajes XMPP**. Un mensaje tiene:
- `sender`: quién lo envía (JID)
- `to`: destinatario (JID)  
- `body`: contenido (string, normalmente JSON o bytes serializados)
- `metadata`: diccionario de etiquetas para filtrar mensajes
- `thread`: hilo de conversación (para relacionar mensajes)

En Royal FLush, los **Templates** filtran mensajes para cada Behaviour:  
```python
# De royalflush/agent/base.py — FlAgent.setup()
Template(metadata={"rf.conversation": "layers"})     # Para LayerReceiver
Template(metadata={"rf.conversation": "similarity"})  # Para SimilarityReceiver
```

In [10]:
import asyncio
import json
from spade.agent import Agent
from spade.behaviour import CyclicBehaviour, OneShotBehaviour
from spade.message import Message
from spade.template import Template

# ──────────────────────────────────────────────────────────────
# DOS AGENTES COMUNICÁNDOSE
# Agente Emisor → envía pesos del modelo
# Agente Receptor → los recibe y hace consenso
# ──────────────────────────────────────────────────────────────

class EmisorBehaviour(OneShotBehaviour):
    """Simula el CommunicationState de Royal FLush."""
    async def run(self):
        await asyncio.sleep(1)  # Espera a que el receptor esté listo
        
        # Simula los pesos de un modelo PyTorch como JSON
        pesos_modelo = {
            "conv1.weight": [0.12, 0.45, -0.23],
            "fc2.weight":   [0.67, -0.11, 0.88]
        }
        
        # Construir mensaje SPADE
        msg = Message(
    to="receptor@localhost",
    body=json.dumps(pesos_modelo),
    metadata={"rf.conversation": "layers"}  # ← solo UNA clave, sin rf.round ni request_reply
)
        """msg = Message(
            to="receptor@localhost",        # Destinatario
            body=json.dumps(pesos_modelo),   # En RF se serializa con pickle/torch.save
            metadata={
                "rf.conversation": "layers",  # ← Template del LayerReceiverBehaviour
                "rf.round": "1",
                "request_reply": "true"
            }
        )"""
        
        await self.send(msg)
        print(f"[emisor] 📤 Pesos enviados a receptor@localhost")
        print(f"[emisor] Payload: {list(pesos_modelo.keys())}")
        await self.agent.stop()


class ReceptorBehaviour(CyclicBehaviour):
    """Simula el LayerReceiverState de Royal FLush."""
    async def run(self):
        # Espera hasta 5s a recibir un mensaje
        msg = await self.receive(timeout=5)
        
        if msg is None:
            print("[receptor] ⏰ Timeout. No llegó ningún mensaje.")
            await self.agent.stop()
            return
        
        # Decodifica el mensaje
        pesos_recibidos = json.loads(msg.body)
        print(f"\n[receptor] 📥 Mensaje recibido de: {msg.sender}")
        print(f"[receptor] Metadata: {msg.metadata}")
        print(f"[receptor] Capas recibidas: {list(pesos_recibidos.keys())}")
        
        # Simula el ConsensusState: promedia los pesos
        mis_pesos = {"conv1.weight": [0.20, 0.30, -0.10], "fc2.weight": [0.50, 0.10, 0.70]}
        consenso = {
            capa: [(a + b) / 2 for a, b in zip(mis_pesos[capa], recibidos)]
            for capa, recibidos in pesos_recibidos.items()
        }
        print(f"[receptor] 🤝 Consenso calculado:")
        for capa, pesos in consenso.items():
            pesos_fmt = [f"{p:.3f}" for p in pesos]
            print(f"  {capa}: {pesos_fmt}")
        
        await self.agent.stop()


class AgenteEmisor(Agent):
    async def setup(self):
        self.add_behaviour(EmisorBehaviour())

class AgenteReceptor(Agent):
    async def setup(self):
        # El Template filtra: sólo procesa mensajes con rf.conversation=="layers"
        # Igual que en Royal FLush con ParallelLayerReceiverBehaviour
        #template = Template(metadata={"rf.conversation": "layers"})
        self.add_behaviour(ReceptorBehaviour())


async def main_comunicacion(): #NO funciona el gather await until finished
    emisor   = AgenteEmisor("emisor@localhost",    "pass1")
    receptor = AgenteReceptor("receptor@localhost", "pass2")
    
    await receptor.start(auto_register=True)
    await emisor.start(auto_register=True)
    
    # Función auxiliar que espera a que un agente termine
    async def esperar(agente, nombre):
        while agente.is_alive():
            await asyncio.sleep(1)
        print(f"🏁 {nombre} finalizado.")
    
    # Espera a que ambos terminen EN PARALELO
    await asyncio.gather(
        esperar(emisor,   "Emisor"),
        esperar(receptor, "Receptor"),
    )
    print("\n✅ Experimento completado.")

await main_comunicacion()

[receptor] ⏰ Timeout. No llegó ningún mensaje.


No behaviour matched for message: <message to="receptor@localhost" from="emisor@localhost" thread="None" metadata={'rf.conversation': 'layers'}>
{"conv1.weight": [0.12, 0.45, -0.23], "fc2.weight": [0.67, -0.11, 0.88]}
</message>


[emisor] 📤 Pesos enviados a receptor@localhost
[emisor] Payload: ['conv1.weight', 'fc2.weight']
🏁 Receptor finalizado.
🏁 Emisor finalizado.

✅ Experimento completado.


---
# PARTE 4 — FSM: la Máquina de Estados de PACoL 🔁

## ¿Qué es una FSM en SPADE?

Una FSM (Finite State Machine) es un `Behaviour` especial donde el agente pasa por  
**estados** y **transiciones** definidas. En Royal FLush, el algoritmo PACoL usa exactamente esto:

```
   ┌─────────────┐
   │  TrainState │ ←────────────────────┐
   └──────┬──────┘                      │
          │ (siempre)                   │
          ▼                             │
  ┌──────────────────┐                  │
  │CommunicationState│                  │
  └────────┬─────────┘                  │
           │                            │
      ┌────┴────┐                       │
      │         │                       │
      ▼         ▼                       │
   [train]  ┌────────────────┐          │
      │     │ ConsensusState │          │
      │     └───────┬────────┘          │
      │             │                   │
      │        ┌────┴────┐              │
      │        │         │              │
      │      [comm]    [train]──────────┘
      └────────►
```

Este es el diagrama real de [`royalflush/behaviour/ppremiofl/fsm.py`]

In [ ]:
import asyncio
import random
from spade.agent import Agent
from spade.behaviour import FSMBehaviour, State

# ──────────────────────────────────────────────────────────────
# FSM que replica la estructura de PACoL en Royal FLush
# ──────────────────────────────────────────────────────────────

# Nombres de los estados (constantes)
ESTADO_TRAIN      = "train"
ESTADO_COMM       = "communication"
ESTADO_CONSENSUS  = "consensus"


class TrainState(State):
    """
    Equivale a royalflush/behaviour/ppremiofl/train.py
    Entrena el modelo local y siempre transiciona a CommunicationState.
    """
    async def run(self):
        agente = self.agent
        print(f"\n[{agente.jid.node}][Ronda {agente.ronda}] 🧠 TRAIN: Entrenando...")
        await asyncio.sleep(0.3)  # Simula entrenamiento con PyTorch
        
        # Actualiza los pesos del modelo del agente
        agente.pesos["fc1"] += 0.01
        agente.accuracy = min(0.99, agente.accuracy + random.uniform(0.01, 0.05))
        
        print(f"[{agente.jid.node}][Ronda {agente.ronda}] ✅ TRAIN: Accuracy = {agente.accuracy:.3f}")
        
        # Comprueba si hemos alcanzado el máximo de rondas
        if agente.ronda >= agente.max_rondas:
            print(f"[{agente.jid.node}] 🏁 Fin del experimento.")
            await agente.stop()
            return
        
        agente.ronda += 1
        # Transición SIEMPRE a Communication
        self.set_next_state(ESTADO_COMM)


class CommunicationState(State):
    """
    Equivale a royalflush/behaviour/ppremiofl/communication.py
    Decide si ir a Consensus o volver a Train.
    """
    async def run(self):
        agente = self.agent
        print(f"[{agente.jid.node}][Ronda {agente.ronda}] 📤 COMM: Enviando pesos a vecino...")
        await asyncio.sleep(0.1)  # Simula envío por XMPP
        
        # En Royal FLush se comprueba si hay mensajes en la cola
        hay_mensajes_pendientes = agente.iter_consenso < agente.max_iter_consenso
        
        if hay_mensajes_pendientes:
            agente.iter_consenso += 1
            print(f"[{agente.jid.node}][Ronda {agente.ronda}] → Transición a CONSENSUS (iter {agente.iter_consenso})")
            self.set_next_state(ESTADO_CONSENSUS)
        else:
            agente.iter_consenso = 0
            print(f"[{agente.jid.node}][Ronda {agente.ronda}] → Transición a TRAIN (fin del consenso)")
            self.set_next_state(ESTADO_TRAIN)


class ConsensusState(State):
    """
    Equivale a royalflush/behaviour/ppremiofl/consensus.py
    Fusiona los modelos recibidos con el modelo local.
    """
    async def run(self):
        agente = self.agent
        print(f"[{agente.jid.node}][Ronda {agente.ronda}] 🤝 CONSENSUS: Fusionando modelos...")
        await asyncio.sleep(0.1)  # Simula el promedio de tensores
        
        # Simula consenso (en RF: promedio ponderado de state_dicts de PyTorch)
        peso_vecino_simulado = agente.pesos["fc1"] + random.uniform(-0.02, 0.02)
        agente.pesos["fc1"] = (agente.pesos["fc1"] + peso_vecino_simulado) / 2
        
        print(f"[{agente.jid.node}][Ronda {agente.ronda}] ✅ CONSENSUS: fc1 = {agente.pesos['fc1']:.4f}")
        
        # Decide: más consenso o volver a entrenar
        if agente.iter_consenso < agente.max_iter_consenso:
            self.set_next_state(ESTADO_COMM)
        else:
            self.set_next_state(ESTADO_TRAIN)


class PacolFSMBehaviour(FSMBehaviour):
    """
    Equivale a royalflush/behaviour/ppremiofl/fsm.py → ParallelPremioFsmBehaviour
    """
    def setup(self):
        # Registra los estados
        self.add_state(name=ESTADO_TRAIN,     state=TrainState(),      initial=True)
        self.add_state(name=ESTADO_COMM,      state=CommunicationState())
        self.add_state(name=ESTADO_CONSENSUS, state=ConsensusState())
        
        # Define las transiciones permitidas
        self.add_transition(source=ESTADO_TRAIN,     dest=ESTADO_COMM)
        self.add_transition(source=ESTADO_COMM,      dest=ESTADO_TRAIN)
        self.add_transition(source=ESTADO_COMM,      dest=ESTADO_CONSENSUS)
        self.add_transition(source=ESTADO_CONSENSUS, dest=ESTADO_COMM)
        self.add_transition(source=ESTADO_CONSENSUS, dest=ESTADO_TRAIN)
    
    async def on_start(self):
        print(f"[{self.agent.jid.node}] 🚀 FSM iniciada.")
    
    async def on_end(self):
        print(f"[{self.agent.jid.node}] 🏁 FSM terminada.")


class AgentePACoL(Agent):
    """Agente que simula un Node de Royal FLush con el algoritmo PACoL."""
    
    async def setup(self):
        # Estado interno del agente (en RF: ModelManager, ConsensusManager...)
        self.ronda          = 1
        self.max_rondas     = 4
        self.iter_consenso  = 0
        self.max_iter_consenso = 2       # En RF: ⌊n/2⌋ donde n = nº de agentes
        self.pesos          = {"fc1": 0.5, "fc2": 0.3}  # En RF: state_dict() de PyTorch
        self.accuracy       = 0.10       # Accuracy inicial aleatoria
        
        # Añade la FSM de PACoL
        fsm = PacolFSMBehaviour()
        self.add_behaviour(fsm)


async def main_fsm():
    agente = AgentePACoL("nodo1@localhost", "password123")
    await agente.start(auto_register=True)
    while agente.is_alive():
        await asyncio.sleep(1)
    
    print("🏁 Agente finalizado.")#await agente.wait_until_finished()

await main_fsm()

---
# PARTE 5 — Mapa del código real de Royal FLush 🗺️

Ahora que entiendes los bloques, vamos a ver cómo encajan en el código real.

In [ ]:
# Este bloque NO necesita servidor XMPP — solo muestra la estructura

mapa = """
╔══════════════════════════════════════════════════════════════════╗
║              MAPA DEL CÓDIGO DE ROYAL FLUSH                      ║
╠══════════════════════════════════════════════════════════════════╣
║                                                                   ║
║  royalflush/                                                      ║
║  │                                                                ║
║  ├── agent/                                                       ║
║  │   ├── base.py         AgentBase, AgentNode, FlAgent            ║
║  │   │                   ↑ Equivale a tu clase Agent de SPADE     ║
║  │   ├── coordinator.py  CoordinatorAgent (sincroniza el inicio)  ║
║  │   ├── observer.py     ObserverAgent (solo registra métricas)   ║
║  │   └── algorithms/                                              ║
║  │       ├── acol.py     AcolAgent (_select_neighbours aleatorio) ║
║  │       └── macofl.py   MAcoFL (selección por similitud)         ║
║  │                                                                ║
║  ├── behaviour/                                                   ║
║  │   ├── ppremiofl/                                               ║
║  │   │   ├── fsm.py          ParallelPremioFsmBehaviour  ← FSM   ║
║  │   │   ├── train.py        TrainState          ← Estado 1       ║
║  │   │   ├── communication.py CommunicationState ← Estado 2       ║
║  │   │   ├── consensus.py    ConsensusState       ← Estado 3       ║
║  │   │   ├── layer_receiver.py   CyclicBehaviour (2ª FSM)        ║
║  │   │   └── similarity_receiver.py CyclicBehaviour              ║
║  │   └── coordination/      FSMs de init (Coordinator, Node...)  ║
║  │                                                                ║
║  ├── core/                                                        ║
║  │   ├── models/                                                  ║
║  │   │   ├── cnn.py     CNN5 (el modelo del paper)                ║
║  │   │   └── mlp.py     MLP sencillo                              ║
║  │   ├── datasets/                                                ║
║  │   │   ├── cifar.py   CIFAR-10 con distribución IID/non-IID     ║
║  │   │   └── mnist.py   MNIST                                     ║
║  │   ├── consensus.py   Operación de promedio de pesos            ║
║  │   └── similarity.py  Cálculo de vectores de similitud          ║
║  │                                                                ║
║  ├── log/               Sistema de logging en CSV                 ║
║  ├── config/            Lectura del fichero YAML (e01.yml)        ║
║  ├── wire/              Serialización de mensajes (multipart)     ║
║  └── runner/                                                      ║
║      └── dispatcher.py  Lanza el experimento (TODO incompleto)    ║
║                                          ↑                        ║
║                                    Aquí está el trabajo pendiente ║
╚══════════════════════════════════════════════════════════════════╝
"""
print(mapa)

In [ ]:
import sys
import os

# ──────────────────────────────────────────────────────────────
# Explorar el código real de Royal FLush
# Ajusta esta ruta a donde hayas descomprimido el ZIP
# ──────────────────────────────────────────────────────────────

RUTA_ROYALFLUSH = "../royalflush_code"  # ← Cambia esto si es necesario

if os.path.exists(RUTA_ROYALFLUSH):
    sys.path.insert(0, RUTA_ROYALFLUSH)
    print(f"✅ Royal FLush encontrado en: {os.path.abspath(RUTA_ROYALFLUSH)}")
else:
    print(f"⚠️  No se encontró Royal FLush en: {RUTA_ROYALFLUSH}")
    print("   Ajusta la variable RUTA_ROYALFLUSH")

In [ ]:
# ──────────────────────────────────────────────────────────────
# Inspeccionar los modelos reales de Royal FLush (sin XMPP)
# ──────────────────────────────────────────────────────────────
import torch
from royalflush.core.models.cnn import CNN5  # El modelo del paper
from royalflush.core.models.mlp import MLP

print("=" * 60)
print("CNN5 — el modelo usado en el experimento PACoL")
print("=" * 60)
cnn = CNN5(num_classes=10)
print(cnn)

# Número de parámetros del modelo
total_params = sum(p.numel() for p in cnn.parameters())
print(f"\nTotal parámetros: {total_params:,}")

# Simula un forward pass con una imagen CIFAR-10 (3x32x32)
imagen_ejemplo = torch.randn(1, 3, 32, 32)  # batch_size=1
with torch.no_grad():
    salida = cnn(imagen_ejemplo)
print(f"Forma de la salida: {salida.shape}  (1 imagen, 10 clases)")
print(f"Logits: {salida[0].tolist()}")

In [ ]:
# ──────────────────────────────────────────────────────────────
# Inspeccionar el consenso real de Royal FLush
# ──────────────────────────────────────────────────────────────
import torch
from royalflush.core.consensus import Consensus
from royalflush.core.models.mlp import MLP

print("Simulando el consenso de dos agentes...")
print("=" * 50)

# Crea dos modelos con pesos diferentes
modelo_a = MLP(input_size=784, hidden_size=128, num_classes=10)
modelo_b = MLP(input_size=784, hidden_size=128, num_classes=10)

# Coge un peso concreto para ver cómo evoluciona
capa = "fc1.weight"
peso_a_antes = modelo_a.state_dict()[capa][0, 0].item()
peso_b       = modelo_b.state_dict()[capa][0, 0].item()

print(f"Agente A — {capa}[0,0] = {peso_a_antes:.6f}")
print(f"Agente B — {capa}[0,0] = {peso_b:.6f}")

# Simula que el Agente A recibe el modelo del Agente B y hace consenso
# En Royal FLush esto lo hace ConsensusState usando ConsensusManager
with torch.no_grad():
    for nombre_capa in modelo_a.state_dict():
        modelo_a.state_dict()[nombre_capa].add_(modelo_b.state_dict()[nombre_capa])
        modelo_a.state_dict()[nombre_capa].div_(2)

peso_a_despues = modelo_a.state_dict()[capa][0, 0].item()
print(f"\nTras consenso:")
print(f"Agente A — {capa}[0,0] = {peso_a_despues:.6f}")
print(f"Promedio esperado     = {(peso_a_antes + peso_b) / 2:.6f}")
print("✅ ¡Coincide!")

---
# PARTE 6 — Ejercicios propuestos 📝

Estos ejercicios te preparan para continuar el TFG.

In [ ]:
# ──────────────────────────────────────────────────────────────
# EJERCICIO 1 (Fácil)
# Modifica el AgentePACoL de la Parte 4 para que imprima
# la accuracy ACTUAL en cada estado Train.
# Pista: accede con self.agent.accuracy
# ──────────────────────────────────────────────────────────────

# Tu código aquí:
pass

In [ ]:
# ──────────────────────────────────────────────────────────────
# EJERCICIO 2 (Medio)
# Crea DOS AgentePACoL que se comuniquen entre sí.
# Cuando llegues al CommunicationState, en lugar de simular,
# envía un mensaje SPADE real con los pesos del modelo.
# ──────────────────────────────────────────────────────────────

# Tu código aquí:
pass

In [ ]:
# ──────────────────────────────────────────────────────────────
# EJERCICIO 3 (Avanzado — conecta con el TFG)
# Mira el archivo royalflush/runner/dispatcher.py
# Verás que _run_local() tiene un TODO:
#   # TODO: launch CoordinatorAgent, ObserverAgents, LauncherAgent, NodeAgents
#
# Tu misión es entender qué hace __build_coordinator()
# e intentar completar __run_local() para que también
# lance los NodeAgents.
# ──────────────────────────────────────────────────────────────

# Primero, lee el código:
import inspect
from royalflush.runner.dispatcher import RunnerDispatcher

# Muestra el código fuente del método que debes completar
print(inspect.getsource(RunnerDispatcher._run_local))

---
# 📚 Resumen — ¿Qué has aprendido?

| Concepto | Donde lo ves en Royal FLush |
|---|---|
| `async def` / `await` | En todos los métodos de cada Behaviour |
| `asyncio.Queue` | `FlAgent.consensus_transmissions` en `base.py` |
| `Agent` + `setup()` | `AgentBase` y `AgentNode` en `agent/base.py` |
| `OneShotBehaviour` | Fase de coordinación inicial |
| `CyclicBehaviour` | `LayerReceiverBehaviour` (siempre escuchando) |
| `FSMBehaviour` | `ParallelPremioFsmBehaviour` en `behaviour/ppremiofl/fsm.py` |
| `Message` + `Template` | Filtrado por `rf.conversation` en `FlAgent` |
| Consenso | Promedio de `state_dict()` en `core/consensus.py` |

---

## Próximos pasos

1. **Instala PyJabber** y ejecuta todas las celdas que necesitan XMPP
2. **Explora** `royalflush/agent/algorithms/acol.py` — es el algoritmo más simple
3. **Lee** `royalflush/runner/dispatcher.py` — ahí están los `TODO` del TFG
4. **Pregunta** a tu tutor cuál es el objetivo concreto de tu TFG

> 💡 **Tip:** Si quieres ejecutar un experimento completo de prueba, usa:  
> ```bash  
> cd royalflush_code  
> royalflush run local e01.yml  
> ```